# BMP Data Cleaning: Iowa NRS Tracking (full export)

Cleans the **full Iowa Nutrient Reduction Strategy (INRS) tracking export** — a
single long, heterogeneous table where each `assessment` (HUC-8 practice
adoption, edge-of-field monitoring, farmer surveys, funding, education events,
modelled load reductions, …) uses a *different subset* of the 37 columns.

**Input:**  `data/tabular/01_raw/bmp/iowa-nrs-tracking.csv`
**Output:** `data/tabular/02_clean/bmp/iowa-nrs-tracking-clean.csv`

Because the file mixes many assessment types, we **keep it in tidy long form**
rather than forcing it wide — the cleaning here is about types, identifiers and
dead weight, not reshaping. Downstream code should filter on `assessment` /
`measurableIndicator` to pull out the slice it needs (the HUC-8 adoption slice
is also available pre-tidied in `iowa-nrs-bmp-huc8-clean.ipynb`).

**Pipeline**
1. Load every column as text (mixed-type columns, leading-zero ids).
2. **Drop fully-empty columns** — 11 of 37 are blank on every row.
3. **Trim & normalise blanks** — strip whitespace, map empty strings to `NaN`.
4. **Parse the measures** — strip thousands separators from `value` -> float;
   `year` -> nullable int.
5. **Normalise watershed / county identifiers** to canonical zero-padded
   strings, and **null the Excel-corrupted `huc12` codes** that were saved in
   scientific notation (`1.02E+11`) and have lost the precision needed to
   recover a unique 12-digit code.
6. Sanity-check and save.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "bmp"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "bmp"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/bmp
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/bmp


## Step 1 — Load

~20K rows. Every column is read as text: the file interleaves numbers, codes and
free text, and several identifiers (`huc8`, `huc12`, `countyFIPS`) are
leading-zero-sensitive and must not be coerced to integers on load.

In [2]:
df = pd.read_csv(RAW_DIR / "iowa-nrs-tracking.csv", dtype="string")
n_raw = len(df)
print(f"Loaded {n_raw:,} rows x {df.shape[1]} columns")
print(f"Distinct assessments: {df['assessment'].nunique()}")
df.head()

Loaded 20,428 rows x 37 columns
Distinct assessments: 45


,measurableIndicator,assessment,referencePeriod,year,value,unit,category,subcategory,tertiarycategory,practiceName,...,cityName,cityFIP,priorityBasinINRSdesignation,priorityBasinINRSname,priorityBasinINRSid,fundingType,fundingCategory,fundingAgency,Order,dataSource
0,Reference,Baseline,Baseline,<NA>,"278,852.00",tons/year,Nitrogen - Nonpoint,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,INRS Baseline
1,Reference,Baseline,Baseline,<NA>,"13,170.00",tons/year,Nitrogen - Point,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,INRS Baseline
2,Reference,Baseline,Baseline,<NA>,"292,022.00",tons/year,Nitrogen - Total,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,INRS Baseline
3,Reference,Baseline,Baseline,<NA>,"21,436.00",tons/year,Phosphorus - Nonpoint,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,INRS Baseline
4,Reference,Baseline,Baseline,<NA>,"2,386.00",tons/year,Phosphorus - Point,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,INRS Baseline


## Step 2 — Drop fully-empty columns

11 of the 37 columns (e.g. `practiceCode`, `mlra`, `Order`, the
`priorityBasinINRS*` trio) are blank on **every** row in this export. We detect
and drop them dynamically so the cleaner stays correct if a future re-pull
populates some of them.

In [3]:
empty_cols = [c for c in df.columns if df[c].notna().sum() == 0]
print(f"Dropping {len(empty_cols)} fully-empty columns:")
for c in empty_cols:
    print("  -", c)
df = df.drop(columns=empty_cols)
print(f"\nRemaining columns: {df.shape[1]}")

Dropping 11 fully-empty columns:
  - practiceCode
  - mlra
  - mlraName
  - watershedProjectID
  - cityName
  - cityFIP
  - priorityBasinINRSdesignation
  - priorityBasinINRSname
  - priorityBasinINRSid
  - fundingType
  - Order

Remaining columns: 26


## Step 3 — Trim whitespace & normalise blanks

Strip surrounding whitespace from every text field and map the resulting empty
strings to `NaN`, so "missing" is represented one way throughout the table.

In [4]:
df = df.apply(lambda s: s.str.strip())
df = df.replace("", pd.NA)
print("Cells normalised. Non-null counts per column:")
print(df.notna().sum().sort_values(ascending=False).to_string())

Cells normalised. Non-null counts per column:
measurableIndicator      20428
unit                     20428
assessment               20428
category                 20428
dataSource               20428
value                    20373
year                     18377
subcategory              15049
countyName               11670
countyFIPS               11670
huc8                      2697
huc8Name                  2697
practiceName              2666
lAMonitoringBasinName     2112
lAMonitoringBasinID       2112
referencePeriod           2078
n                         1865
tertiarycategory          1801
huc12                     1518
huc12Name                 1518
fundingCategory            334
fundingAgency              334
species                    192
poundsPerAcre              192
cropRotation               192
watershedProjectName        60


## Step 4 — Parse the measures

`value` carries `,` thousands separators (e.g. `"278,852.00"`); strip them and
coerce to float. `year` becomes a nullable integer. The free-text `n` (sample
size / survey notes) is intentionally **left as a string** — it mixes counts
with descriptive text and isn't a clean numeric.

In [5]:
df["value"] = pd.to_numeric(
    df["value"].str.replace(",", "", regex=False), errors="coerce"
)
df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")

print("value missing:", int(df["value"].isna().sum()),
      f"({df['value'].isna().mean():.1%})")
print("year  missing:", int(df["year"].isna().sum()),
      f"({df['year'].isna().mean():.1%})")
print("year range:", int(df["year"].min()), "->", int(df["year"].max()))

value missing: 55 (0.3%)
year  missing: 2051 (10.0%)
year range: 1978 -> 2023


## Step 5 — Normalise watershed & county identifiers

Re-pad the integer-truncated codes to their canonical widths (`huc8` -> 8,
`huc12` -> 12, `countyFIPS` -> 5).

A subset of `huc12` codes were exported from Excel in **scientific notation**
(`1.02E+11`) and have lost the low-order digits — there is no way to recover a
unique 12-digit code from them, so we **null the corrupted codes** rather than
fabricate one. The accompanying `huc12Name` is retained, so those rows still
carry their watershed identity by name.

In [6]:
# Excel mangled some HUC-12s into scientific notation; these are unrecoverable.
sci = df["huc12"].notna() & df["huc12"].str.contains("E", case=False, na=False)
print(f"Nulling {int(sci.sum())} scientific-notation-corrupted huc12 codes "
      f"(huc12Name kept)")
df.loc[sci, "huc12"] = pd.NA

PAD = {"huc8": 8, "huc12": 12, "countyFIPS": 5}
for col, width in PAD.items():
    df[col] = df[col].str.zfill(width)
    lens = sorted(df[col].dropna().str.len().unique())
    print(f"{col:11s} -> widths {lens}, "
          f"{df[col].nunique()} distinct, {int(df[col].notna().sum())} non-null")

Nulling 385 scientific-notation-corrupted huc12 codes (huc12Name kept)
huc8        -> widths [np.int64(8)], 56 distinct, 2697 non-null
huc12       -> widths [np.int64(12)], 484 distinct, 1133 non-null
countyFIPS  -> widths [np.int64(5)], 99 distinct, 11670 non-null


## Step 6 — Sanity check

Confirm no exact duplicate rows survived, and summarise the cleaned table by its
top-level `measurableIndicator` so the long-form structure is visible at a
glance.

In [7]:
n_dupes = int(df.duplicated().sum())
print(f"Exact duplicate rows: {n_dupes}")
print(f"Final shape: {df.shape[0]:,} rows x {df.shape[1]} columns "
      f"({df.shape[0] / n_raw:.0%} of raw rows)\n")
print("Rows by measurableIndicator:")
print(df["measurableIndicator"].value_counts(dropna=False).to_string())
print("\nTop assessments:")
print(df["assessment"].value_counts().head(10).to_string())

Exact duplicate rows: 2
Final shape: 20,428 rows x 26 columns (100% of raw rows)

Rows by measurableIndicator:
measurableIndicator
Human        13984
Land          3992
Water         2010
Input          436
Reference        6

Top assessments:
assessment
Education & Outreach - Events per Year                                                   11912
HUC8 Practice Adoption                                                                    2643
Statewide Monitoring by HUC12                                                             1518
Farmer Survey - Influence of sources on nutrient management decisions                      425
Farmer Survey - Farmer perspectives on barriers on water quality improvements in Iowa      375
Farmer Survey - Farmer perspectives on INRS topics                                         350
Funding - Principal Programs                                                               334
Farmer Survey - Farmers who indicated they used the practice in first survey    

## Step 7 — Save

In [8]:
out_file = CLEAN_DIR / "iowa-nrs-tracking-clean.csv"
df.to_csv(out_file, index=False)
print(f"Saved {len(df):,} rows x {df.shape[1]} columns -> {out_file}")

Saved 20,428 rows x 26 columns -> /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/bmp/iowa-nrs-tracking-clean.csv
